# FATO_PRODUCAO_AMBULATORIAL

Fato da produção ambulatorial (SIA). **Grão:** competência × município de residência ×
município de atendimento × estabelecimento × procedimento.

Os registros do SIA que caem na mesma combinação de chaves são **somados**. Assim a PK
é formada exatamente pelas 5 FKs, e o volume cai de ~13 milhões de registros para
alguns milhões de linhas, sem perder nenhum valor.

- **Medidas (todas SUM):** `QTD_REGISTROS`, `QTD_APRESENTADA`, `QTD_APROVADA`,
  `VL_APRESENTADO`, `VL_APROVADO`. Os indicadores usam os valores **aprovados**.
- Tabela particionada por mês (`SK_TEMPO_COMPETENCIA`).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar, coletar_estatisticas

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO FATO_PRODUCAO_AMBULATORIAL
    (SK_TEMPO_COMPETENCIA, SK_MUNICIPIO_RESIDENCIA, SK_MUNICIPIO_ATENDIMENTO,
     SK_ESTABELECIMENTO, SK_PROCEDIMENTO, QTD_REGISTROS, QTD_APRESENTADA, QTD_APROVADA,
     VL_APRESENTADO, VL_APROVADO)
SELECT NVL(t.SK_TEMPO, -1),
       NVL(mr.SK_MUNICIPIO, -1),
       NVL(ma.SK_MUNICIPIO, -1),
       NVL(e.SK_ESTABELECIMENTO, -1),
       NVL(p.SK_PROCEDIMENTO, -1),
       COUNT(*),
       SUM(s.QTD_APRESENTADA),
       SUM(s.QTD_APROVADA),
       SUM(s.VL_APRESENTADO),
       SUM(s.VL_APROVADO)
FROM SILVER_SIA s
LEFT JOIN DIM_TEMPO t           ON t.DT_COMPLETA = s.DT_COMPETENCIA
LEFT JOIN DIM_MUNICIPIO mr      ON mr.COD_IBGE_6 = s.COD_IBGE_6_RES
LEFT JOIN DIM_MUNICIPIO ma      ON ma.COD_IBGE_6 = s.COD_IBGE_6_ATEND
LEFT JOIN DIM_ESTABELECIMENTO e ON e.COD_CNES = s.COD_CNES
LEFT JOIN DIM_PROCEDIMENTO p    ON p.COD_PROCEDIMENTO = s.COD_PROCEDIMENTO
GROUP BY NVL(t.SK_TEMPO, -1), NVL(mr.SK_MUNICIPIO, -1), NVL(ma.SK_MUNICIPIO, -1),
         NVL(e.SK_ESTABELECIMENTO, -1), NVL(p.SK_PROCEDIMENTO, -1)
"""

In [ ]:
recarregar("FATO_PRODUCAO_AMBULATORIAL", SQL, origem="SILVER_SIA")
coletar_estatisticas("FATO_PRODUCAO_AMBULATORIAL")

In [ ]:
consultar("SELECT * FROM FATO_PRODUCAO_AMBULATORIAL FETCH FIRST 10 ROWS ONLY")